In [0]:
# =====================================================================================
# Notebook: 02_silver/02_dim_date_and_fact_game_enriched.py
# Project : nba-datalakehouse
# Purpose : Build additional Silver tables:
#           1) dim_date (calendar dimension)
#           2) fact_game_enriched (fact_game joined with team names)
#
# Why these matter:
# - dim_date: standard warehouse pattern; simplifies time-based analytics and BI
# - fact_game_enriched: makes querying easier (no repeated joins in every analysis)
#
# V1 approach:
# - rebuild tables each run (overwrite)
# - later we can incrementalize dim_date by expanding date range
# =====================================================================================

from pyspark.sql import functions as F

SILVER_DIM_TEAM = "workspace.nba_silver.dim_team"
SILVER_FACT_GAME = "workspace.nba_silver.fact_game"

SILVER_DIM_DATE = "workspace.nba_silver.dim_date"
SILVER_FACT_GAME_ENRICHED = "workspace.nba_silver.fact_game_enriched"

# -----------------------------
# 0) Load upstream Silver tables
# -----------------------------
dim_team = spark.table(SILVER_DIM_TEAM)
fact_game = spark.table(SILVER_FACT_GAME)

# -----------------------------
# 1) Build dim_date
# -----------------------------
# We want a date dimension that covers the full span of your fact data, plus some buffer.
# We'll derive the min/max date from fact_game.game_date_utc.
#
# dim_date grain: 1 row per calendar date (date_key)

min_max = fact_game.select(
    F.min("game_date_utc").alias("min_date"),
    F.max("game_date_utc").alias("max_date"),
).collect()[0]

min_date = min_max["min_date"]
max_date = min_max["max_date"]

buffer_days = 365
start = min_date.toordinal() - buffer_days
end = max_date.toordinal() + buffer_days

dim_date_df = (
    spark.range(start, end + 1)
    .select(F.expr("date_add('0001-01-01', cast(id as int) - 1)").alias("date_key"))
    .withColumn("year", F.year("date_key"))
    .withColumn("month", F.month("date_key"))
    .withColumn("day", F.dayofmonth("date_key"))
    .withColumn("day_of_week_num", F.dayofweek("date_key"))
    .withColumn("day_name", F.date_format("date_key", "EEEE"))
    .withColumn("month_name", F.date_format("date_key", "MMMM"))
    .withColumn("week_of_year", F.weekofyear("date_key"))
    .withColumn("quarter", F.quarter("date_key"))
    .withColumn("is_weekend", F.col("day_of_week_num").isin([1, 7]).cast("boolean"))
)


spark.sql(f"DROP TABLE IF EXISTS {SILVER_DIM_DATE}")

(dim_date_df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable(SILVER_DIM_DATE))

print("Wrote:", SILVER_DIM_DATE, "rows=", dim_date_df.count())
display(dim_date_df.orderBy(F.col("date_key").desc()).limit(10))

# -----------------------------
# 2) Build fact_game_enriched
# -----------------------------
# Adds team names and a few convenience fields.
# Grain remains 1 row per game_id_bdl.

fact_game_enriched = (
    fact_game.alias("g")
    .join(dim_team.alias("ht"), F.col("g.home_team_id_bdl") == F.col("ht.team_id_bdl"), "left")
    .join(dim_team.alias("vt"), F.col("g.visitor_team_id_bdl") == F.col("vt.team_id_bdl"), "left")
    .select(
        F.col("g.game_id_bdl"),
        F.col("g.season_start_year"),
        F.col("g.game_date_utc"),
        F.col("g.start_time_utc"),
        F.col("g.status"),
        F.col("g.game_state"),
        F.col("g.postseason"),
        F.col("g.period"),
        F.col("g.time_display"),
        F.col("g.home_team_id_bdl"),
        F.col("ht.full_name").alias("home_team_name"),
        F.col("g.visitor_team_id_bdl"),
        F.col("vt.full_name").alias("visitor_team_name"),
        F.col("g.home_score"),
        F.col("g.visitor_score"),
        (F.col("g.home_score") - F.col("g.visitor_score")).alias("home_margin"),
        F.col("g.source_dt"),
        F.col("g.ingested_at"),
        F.col("g.source_system"),
    )
)

spark.sql(f"DROP TABLE IF EXISTS {SILVER_FACT_GAME_ENRICHED}")

(fact_game_enriched.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("season_start_year")
 .saveAsTable(SILVER_FACT_GAME_ENRICHED))

print("Wrote:", SILVER_FACT_GAME_ENRICHED, "rows=", fact_game_enriched.count())
display(fact_game_enriched.orderBy(F.col("start_time_utc").desc()).limit(20))

# -----------------------------
# 3) Quick sanity checks
# -----------------------------
print("\n=== Sanity: any missing team names? ===")
fact_game_enriched.select(
    F.sum(F.when(F.col("home_team_name").isNull(), 1).otherwise(0)).alias("missing_home_team_name"),
    F.sum(F.when(F.col("visitor_team_name").isNull(), 1).otherwise(0)).alias("missing_visitor_team_name"),
    F.count("*").alias("total_games"),
).show(truncate=False)

print("\n=== Sample: most recent scheduled games ===")
fact_game_enriched.filter(F.col("game_state") == "Scheduled") \
    .orderBy(F.col("start_time_utc").desc()) \
    .select("game_date_utc","start_time_utc","home_team_name","visitor_team_name","status","game_state","source_dt") \
    .show(25, truncate=False)
